---
title: Loading data from the Open Data Cube
short_title: Loading data
subject: Beginner Guide
subtitle: Loading satellite imagery from the datacube with dc.load.
description: Loading satellite imagery from the datacube with dc.load.
authors:
  - name: Muhammad Taufik
    github: taufik-shf
  - name: Alex G Leith
    github: alexgleith
keywords:
  - open-data-cube
  - odc
  - beginner-guide
---

This notebook covers `dc.load`, the function that retrieves satellite imagery from the datacube. It shows how to define a query, load the imagery, and modify the query to load different data.[^edits]

[^edits]: Tutorial notebooks update automatically; edits to a tutorial notebook may be overwritten on the next update. Keep a working copy in a separate file to preserve changes.

## A. Objectives

- Load data with `dc.load`
- Read the returned Dataset
- Find datasets before loading
- Load a specific set of datasets

## B. Loading Data

### 1. Connecting to the datacube

As always, first a connection to the datacube needs to be established. Note the object is stored under a variable called `dc` (dc is convention, any name might work).

In [ ]:
from datacube import Datacube

dc = Datacube(app="loading_data")

### 2. Query + dc.load

After exploring products and measurements in [02_odc_introduction.ipynb](02_odc_introduction.ipynb), the next step is loading actual pixel data with `dc.load`. `dc.load` is a function that takes a query, which commonly includes the product, area, time, and measurements, among other options.

The following cell sets up the query variables.

In [ ]:
query = {
    "product": "s2_geomad_annual",
    "x": (98.80, 98.90),
    "y": (2.65, 2.55),
    "time": "2024",
    "measurements": ["red", "green", "blue"],
    "output_crs": "EPSG:32647",
    "resolution": (-30, 30),
}

The query is set to provide arguments which are expected by `dc.load`, in this case the parameters used are:

- **product**: the product to load from the datacube. Use `dc.list_products()` to see the full list of available products.
- **x**, **y**: longitude and latitude ranges in degrees, each as a tuple of two values. Should fall within the product's spatial extent; queries outside the extent return an empty result.
- **time**: a date or date range. Accepts a year (`"2024"`), a month (`"2024-01"`), a specific date (`"2024-01-15"`), or a tuple for a range (e.g., `("2024-01-01", "2024-06-30")`).
- **measurements**: which measurements of the product to return, as a list. Use `dc.list_measurements()` to see what a product provides.
- **output_crs**: the coordinate reference system to project the result into, specified as an EPSG code (e.g., `"EPSG:32647"` - UTM 47N). Required for products with no default; a local UTM zone is a common choice.
- **resolution**: the pixel size in `output_crs` units, given as `(y, x)`. The y value is typically negative because rows count from north to south.

Other common parameters that might be used in a query include: 

- **datasets**: a list of pre-searched Dataset objects returned by `dc.find_datasets()`. Used with the find-then-load workflow (section E) to load a specific set of scenes.
- **group_by**: how to group multiple observations in the same period. Common value: `"solar_day"`.
- **dask_chunks**: chunk configuration for lazy, parallel loading via Dask. Useful for large loads that won't fit in memory.
- **resampling**: how to resample values when reprojecting or changing resolution. Common values: `"nearest"` (default), `"bilinear"`, `"cubic"`.

The next cell passes the query to `dc.load`, which returns an `xarray.Dataset`. `xarray.Dataset` is a multi-dimensional Python object that is covered in [04_xarray_for_odc.ipynb](04_xarray_for_odc.ipynb).

In [ ]:
ds = dc.load(**query) 
ds # ds is another convention which stands for dataset

## C. Reading the returned Dataset

The `xarray.Dataset` returned by `dc.load` has these parts, each available as an attribute of `ds`:

- **Dimensions**: This header identifies the number of timesteps returned (time: 1) as well as the number of resulting pixels in the x and y direction (y: 369, x: 372).
- **Coordinates**: the labels along each axis. `time` carries the timestamps; `y` and `x` carry projected coordinates in the CRS from `output_crs`; `spatial_ref` describes the CRS itself.
- **Data variables**: These are the measurements available for the loaded product. For every timestep (time) returned by the query, the measured value at each pixel (y, x) is returned as an array for each measurement.

Individual measurements come out as `xarray.DataArray` objects that can be accessed by name:

In [ ]:
ds.red

The loaded data can also be inspected by plotting:

In [ ]:
ds.red.plot.imshow(col="time")

## D. Find datasets

Sometimes it helps to know what data exists before loading. `dc.find_datasets` searches the datacube and returns a list of matching Dataset objects, without loading pixel data. This is useful for checking counts, previewing what a query would match, or selecting a specific subset to feed into `dc.load`.

These `Dataset` objects (metadata handles) are distinct from the `xarray.Dataset` that `dc.load` returns (pixel data).

`dc.find_datasets` takes the same query shape as `dc.load`, minus the output-grid parameters (`output_crs`, `resolution`) and `measurements`:

In [ ]:
datasets = dc.find_datasets(
    product="s2_geomad_annual",
    x=(98.80, 98.90),
    y=(2.65, 2.55),
    time=("2020", "2024"),
)

len(datasets)

Each item in the returned list carries metadata about one underlying scene: file locations, native CRS, measurement paths, and timestamps.

In [ ]:
datasets[0]

## E. Load selected datasets

With a list of datasets from `dc.find_datasets`, `dc.load` can load their pixel data directly. Passing the list via the `datasets` parameter replaces the search: `product`, `x`, `y`, and `time` are read from the datasets themselves. `output_crs`, `resolution`, and `measurements` are still needed.

In [ ]:
ds = dc.load(
    datasets=datasets,
    measurements=["red", "green", "blue"],
    output_crs="EPSG:32647",
    resolution=(-30, 30),
)

ds

The returned `xarray.Dataset` has the same structure as before, but with more timesteps, one per annual composite loaded.

## G. Next steps

The next notebook describes the structure of the `xarray.Dataset` that `dc.load` returns: how to read its dimensions and coordinates, select subsets, and combine variables.

Continue to [04_xarray_for_odc.ipynb](04_xarray_for_odc.ipynb).